# Reusable template — FX lift / conversion logistic GD + sklearn

**Short name:** `FX_Quote_GD_Sklearn`  
Point `DATA_PATH` at any blotter with numeric features and a 0/1 lift-or-convert column.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

DATA_PATH = "data/fx_quote_blotter6.csv"
FEATURE_COLS = ["edge_score", "urgency_score"]
TARGET_COL = "lifted"
ALPHA = 0.1
ITERS = 5000
THRESHOLD = 0.5   # raise to only count high-confidence lifts

df = pd.read_csv(DATA_PATH)
X = df[FEATURE_COLS].to_numpy(dtype=float)
y = df[TARGET_COL].to_numpy(dtype=float)

def sigmoid(z):
    z = np.clip(np.asarray(z, dtype=float), -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-z))

def cost(X, y, w, b):
    f = np.clip(sigmoid(X @ w + b), 1e-15, 1 - 1e-15)
    return float(-np.mean(y * np.log(f) + (1 - y) * np.log(1 - f)))

def grad(X, y, w, b):
    err = sigmoid(X @ w + b) - y
    return float(np.mean(err)), (X.T @ err) / X.shape[0]

w = np.zeros(X.shape[1]); b = 0.0
for i in range(ITERS):
    dj_db, dj_dw = grad(X, y, w, b)
    w = w - ALPHA * dj_dw; b = b - ALPHA * dj_db
    if i % max(ITERS // 10, 1) == 0:
        print(f"iter {i:5d}  J={cost(X, y, w, b):.6f}")

proba = sigmoid(X @ w + b)
print("GD w,b:", w, b, "fill-rate:", np.mean((proba >= THRESHOLD) == y), "J:", cost(X, y, w, b))

clf = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=2000).fit(X, y)
print("sklearn C=inf fill-rate:", clf.score(X, y), "coef:", clf.coef_)
print("sklearn L2 fill-rate:", LogisticRegression().fit(X, y).score(X, y))


Swap in `data/fx_quote_practice.csv` or a live stream extract. Raise `THRESHOLD` for a more conservative sales filter. Cut `ALPHA` if $J$ increases. This template does not place orders.
